# Ladybug + Icebug: in-memory manual-insert Leiden

This notebook builds an in-memory Ladybug graph, exports it to Parquet `EXPORT DATABASE`, imports it into a fresh in-memory instance, extracts CSR with `query_as_arrow(...).csr()`, runs Icebug Leiden, writes the communities back onto the nodes, and queries the updated graph.


## Environment

Dependencies are declared in `pyproject.toml`. From a fresh checkout, start Jupyter with:

```bash
uv run jupyter notebook ladybug_mem_icebug_mem.ipynb
```

Or execute the notebook from the CLI with:

```bash
uv run jupyter nbconvert --to notebook --execute ladybug_mem_icebug_mem.ipynb --inplace
```

In [20]:
import os
import shutil
import tempfile
from pathlib import Path

import icebug as ib
import ladybug
import pandas as pd
import pyarrow as pa

print("ladybug", ladybug.__version__)
print("icebug", ib.__version__)

workdir = Path(tempfile.mkdtemp(prefix="ladybug_import_export_"))
export_dir = workdir / "exported-db"
print("workdir:", workdir)


ladybug 0.17.0
icebug 12.9
workdir: /tmp/ladybug_import_export_cr42q13_


## Build an in-memory Ladybug graph

The graph is intentionally small: two dense groups joined by a couple of bridge edges so Leiden can recover the communities after import.

In [21]:
source_db = ladybug.Database()
source = ladybug.Connection(source_db)

people = [
    {"id": 0, "name": "Ada", "team": "platform", "community": 0},
    {"id": 1, "name": "Ben", "team": "platform", "community": 0},
    {"id": 2, "name": "Cora", "team": "platform", "community": 0},
    {"id": 3, "name": "Drew", "team": "platform", "community": 0},
    {"id": 4, "name": "Eli", "team": "research", "community": 0},
    {"id": 5, "name": "Fay", "team": "research", "community": 0},
    {"id": 6, "name": "Gus", "team": "research", "community": 0},
    {"id": 7, "name": "Ivy", "team": "research", "community": 0},
]
undirected_edges = [
    (0, 1, 9), (0, 2, 8), (0, 3, 7), (1, 2, 8), (1, 3, 7), (2, 3, 9),
    (4, 5, 9), (4, 6, 8), (4, 7, 7), (5, 6, 7), (5, 7, 8), (6, 7, 9),
    (3, 4, 3), (2, 5, 3),
]
edge_rows = []
for src, dst, strength in undirected_edges:
    edge_rows.append({"src": src, "dst": dst, "strength": strength})
    edge_rows.append({"src": dst, "dst": src, "strength": strength})

def cypher_list(rows, keys):
    items = []
    for row in rows:
        fields = []
        for key in keys:
            value = row[key]
            if isinstance(value, str):
                fields.append(f"{key}: '{value}'")
            else:
                fields.append(f"{key}: {int(value)}")
        items.append("{" + ", ".join(fields) + "}")
    return "[" + ", ".join(items) + "]"

source.execute("""
CREATE NODE TABLE Person(
    id INT64,
    name STRING,
    team STRING,
    community INT64,
    PRIMARY KEY(id)
)
""")
source.execute("""
CREATE REL TABLE KNOWS(
    FROM Person TO Person,
    strength INT64
)
""")
source.execute(f"""
UNWIND {cypher_list(people, ['id', 'name', 'team', 'community'])} AS row
CREATE (:Person {{
    id: row.id,
    name: row.name,
    team: row.team,
    community: row.community
}})
""")
source.execute(f"""
UNWIND {cypher_list(edge_rows, ['src', 'dst', 'strength'])} AS row
MATCH (a:Person {{id: row.src}}), (b:Person {{id: row.dst}})
CREATE (a)-[:KNOWS {{strength: row.strength}}]->(b)
""")
source.query_as_arrow("""
MATCH (p:Person)
RETURN p.id AS id, p.name AS name, p.team AS team, p.community AS community
ORDER BY p.id
""", 4096).get_as_arrow().to_pandas()


,id,name,team,community
0,0,Ada,platform,0
1,1,Ben,platform,0
2,2,Cora,platform,0
3,3,Drew,platform,0
4,4,Eli,research,0
5,5,Fay,research,0
6,6,Gus,research,0
7,7,Ivy,research,0


## [NOT REQUIRED; DEMONSTRATION PURPOSES ONLY] Export the database, close it, and import it into a fresh in-memory instance

The export writes Parquet files plus `schema.cypher` into a temporary directory. The source instance is then closed before the next in-memory instance imports that Parquet export.

In [22]:
if os.path.exists(export_dir) and os.path.isdir(export_dir):
    shutil.rmtree(export_dir)

source.execute(f"EXPORT DATABASE '{export_dir}'")
source.close()

print(sorted(p.name for p in export_dir.iterdir()))

source_db = ladybug.Database()
source = ladybug.Connection(source_db)
source.execute(f"IMPORT DATABASE '{export_dir}'")

source.query_as_arrow("""
MATCH (p:Person)
RETURN p.id AS id, p.name AS name, p.team AS team, p.community AS community
ORDER BY p.id
""", 4096).get_as_arrow().to_pandas()


['KNOWS_Person_Person.parquet', 'Person.parquet', 'copy.cypher', 'index.cypher', 'schema.cypher']


,id,name,team,community
0,0,Ada,platform,0
1,1,Ben,platform,0
2,2,Cora,platform,0
3,3,Drew,platform,0
4,4,Eli,research,0
5,5,Fay,research,0
6,6,Gus,research,0
7,7,Ivy,research,0


## Extract CSR from the imported database and run Icebug Leiden

`query_as_arrow(...).csr()` turns the imported rowid projection into CSR arrays, and those arrays feed `Graph.fromCSR` directly.

In [23]:
n_nodes = source.query_as_arrow(
    "MATCH (p:Person) RETURN count(p) AS n",
    1024,
).get_as_arrow().column(0)[0].as_py()

relationship_result = source.query_as_arrow("""
MATCH (a:Person)-[r:KNOWS]->(b:Person)
RETURN a.rowid AS src_rowid,
       r.rowid AS rel_rowid,
       b.rowid AS dst_rowid
ORDER BY src_rowid, rel_rowid
""", 4096)
relationship_csr = relationship_result.csr()

indptr = relationship_csr.indptr.cast(pa.uint64())
indices = relationship_csr.indices.cast(pa.uint64())

graph = ib.Graph.fromCSR(n_nodes, False, indices, indptr)
leiden = ib.community.ParallelLeidenView(graph, iterations=4, gamma=1.0, randomize=False)
leiden.run()
partition = leiden.getPartition()
modularity = ib.community.Modularity().getQuality(partition, graph)

community_frame = source.query_as_arrow("""
MATCH (p:Person)
RETURN p.id AS id, p.name AS name, p.team AS team
ORDER BY p.id
""", 4096).get_as_arrow().to_pandas()
community_frame["community"] = [partition.subsetOf(i) for i in range(graph.numberOfNodes())]

print(f"Icebug found {community_frame['community'].nunique()} communities; modularity Q={modularity:.4f}")
community_frame


Icebug found 2 communities; modularity Q=0.3571


,id,name,team,community
0,0,Ada,platform,0
1,1,Ben,platform,0
2,2,Cora,platform,0
3,3,Drew,platform,0
4,4,Eli,research,1
5,5,Fay,research,1
6,6,Gus,research,1
7,7,Ivy,research,1


## Assign the Icebug communities back to the imported nodes

The imported `Person` table already has a `community` column, so Cypher `SET` can write the Leiden partition back onto each node.

In [24]:
for node_id, community in community_frame[["id", "community"]].itertuples(index=False):
    source.execute(f"MATCH (p:Person {{id: {int(node_id)}}}) SET p.community = {int(community)}")

updated_nodes = source.query_as_arrow("""
MATCH (p:Person)
RETURN p.id AS id, p.name AS name, p.team AS team, p.community AS community
ORDER BY p.community, p.team, p.id
""", 4096).get_as_arrow().to_pandas()

updated_nodes

,id,name,team,community
0,0,Ada,platform,0
1,1,Ben,platform,0
2,2,Cora,platform,0
3,3,Drew,platform,0
4,4,Eli,research,1
5,5,Fay,research,1
6,6,Gus,research,1
7,7,Ivy,research,1


In [25]:
source.query_as_arrow("""
MATCH (a:Person)-[r:KNOWS]->(b:Person)
RETURN a.name AS source,
       a.community AS source_community,
       b.name AS target,
       b.community AS target_community,
       r.strength AS strength
ORDER BY source_community, source, target
LIMIT 12
""", 4096).get_as_arrow().to_pandas()


,source,source_community,target,target_community,strength
0,Ada,0,Ben,0,9
1,Ada,0,Cora,0,8
2,Ada,0,Drew,0,7
3,Ben,0,Ada,0,9
4,Ben,0,Cora,0,8
5,Ben,0,Drew,0,7
6,Cora,0,Ada,0,8
7,Cora,0,Ben,0,8
8,Cora,0,Drew,0,9
9,Cora,0,Fay,1,3


## Clean up temporary notebook data

Remove the exported Parquet directory and close the imported instance once the results have been displayed.

In [26]:
if os.path.exists(export_dir) and os.path.isdir(export_dir):
    shutil.rmtree(export_dir)

source.close()
shutil.rmtree(workdir)
print("temporary data removed:", not workdir.exists())


temporary data removed: True
